# 🎯 미세 조정 (원본 0.5955 기반)

## 교훈
- 캘리브레이션 강화 → 속도 저하 → 점수 하락
- **원본 설정 유지가 핵심**

## 전략
- 원본 설정 100% 유지
- dampening만 미세 조정 (0.001 → 0.0008)

In [1]:
import os
import torch
import shutil
import json

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

PyTorch: 2.9.1
CUDA: False


In [2]:
# ============================================================================
# 원본 0.5955 설정 (그대로 유지!)
# ============================================================================

MODEL_ID = "./open/base_model"
OUT_DIR = "./model"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"

# 캘리브레이션 (원본 그대로!)
NUM_SAMPLES = 256       # 원본 유지
MAX_SEQ_LEN = 512       # 원본 유지

# 양자화 설정 (원본 그대로!)
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]

# 최적화 파라미터 (원본 + 미세조정)
BLOCK_SIZE = 128        # 원본 유지
DAMPENING = 0.0008      # 🎯 0.001 → 0.0008 (미세 조정)
ACTORDER = "weight"     # 원본 유지

print("=" * 60)
print("원본 0.5955 기반 미세 조정")
print("=" * 60)
print(f"캘리브레이션: {NUM_SAMPLES}샘플, {MAX_SEQ_LEN}길이 (원본 유지)")
print(f"dampening: {DAMPENING} (0.001 → 0.0008)")
print("=" * 60)

원본 0.5955 기반 미세 조정
캘리브레이션: 256샘플, 512길이 (원본 유지)
dampening: 0.0008 (0.001 → 0.0008)


In [3]:
print("[1/5] 모델 로드...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"  파라미터: {model.num_parameters():,}")

`torch_dtype` is deprecated! Use `dtype` instead!


[1/5] 모델 로드...
  파라미터: 1,279,391,488


In [4]:
print(f"[2/5] 데이터셋 로드 ({NUM_SAMPLES}개)...")

ds = load_dataset(DATASET_ID, split=f"train[:{NUM_SAMPLES}]")

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)
print(f"  완료: {len(ds)}개")

[2/5] 데이터셋 로드 (256개)...
  완료: 256개


In [ ]:
print("[3/5] GPTQ 양자화...")
print(f"  dampening: {DAMPENING}")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQ_LEN,
    num_calibration_samples=NUM_SAMPLES,
)

print("  완료!")

[3/5] GPTQ 양자화...
  dampening: 0.0008


Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-12T11:16:52.519645+0900 | reset | INFO - Compression lifecycle reset
2026-02-12T11:16:52.521123+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-12T11:16:52.545641+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-12T11:16:52.546034+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-12T11:16:52.552424+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0212 11:16:52.585000 63169 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:57<00:00,  4.49it/s]

2026-02-12T11:17:49.869835+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

2026-02-12T11:29:20.093436+0900 | compress | METRIC - time 0.38s
2026-02-12T11:29:20.093950+0900 | compress | METRIC - error 275.25
2026-02-12T11:29:20.095230+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T11:29:20.095563+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-12T11:29:20.097370+0900 | compress_modules | INFO - Quantizing model.layers.10.self_attn.k_proj using 256 samples
2026-02-12T11:29:20.338247+0900 | compress | METRIC - time 0.24s
2026-02-12T11:29:20.338628+0900 | compress | METRIC - error 74.05
2026-02-12T11:29:20.339682+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T11:29:20.339939+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-12T11:29:20.340663+0900 | compress_modules | INFO - Quantizing model.layers.10.self_attn.v_proj using 256 samples
2026-02-12T11:29:20.592388+0900 | compress | METRIC - time 0.25s
2026-02-12T11:29:

(12/31): Calibrating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:55<00:00,  4.62it/s]

2026-02-12T11:30:28.569703+0900 | compress_modules | INFO - Quantizing model.layers.11.self_attn.q_proj using 256 samples


2026-02-12T11:30:28.953352+0900 | compress | METRIC - time 0.38s
2026-02-12T11:30:28.953862+0900 | compress | METRIC - error 298.21
2026-02-12T11:30:28.955194+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T11:30:28.955469+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-12T11:30:28.957191+0900 | compress_modules | INFO - Quantizing model.layers.11.self_attn.k_proj using 256 samples
2026-02-12T11:30:29.199423+0900 | compress | METRIC - time 0.24s
2026-02-12T11:30:29.199956+0900 | compress | METRIC - error 84.34
2026-02-12T11:30:29.200932+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-12T11:30:29.201194+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-12T11:30:29.202027+0900 | compress_modules | INFO - Quantizing model.layers.11.self_attn.v_proj using 256 samples
2026-02-12T11:30:29.466642+0900 | compress | METRIC - time 0.26s
2026-02-12T11:30:

(12/31): Propagating:  35%|██████████████████████████████████████████████████████▏                                                                                                   | 90/256 [00:03<00:06, 24.85it/s]

In [ ]:
print("[4/5] 모델 저장...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

total = sum(os.path.getsize(os.path.join(OUT_DIR, f)) for f in os.listdir(OUT_DIR))
print(f"  크기: {total/1e9:.2f} GB")

In [ ]:
# 검증
with open(f"{OUT_DIR}/config.json") as f:
    cfg = json.load(f)

print("\n[검증]")
print(f"  tie_word_embeddings: {cfg.get('tie_word_embeddings')}")

if cfg.get('tie_word_embeddings') == True:
    print("✅ vLLM 호환!")

In [ ]:
print("[5/5] 제출 파일 생성...")

zip_name = "submit"
if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(zip_name, "zip", ".", OUT_DIR)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9

print("\n" + "=" * 60)
print("제출 준비 완료!")
print("=" * 60)
print(f"파일: {zip_name}.zip ({zip_size:.2f} GB)")
print("=" * 60)